# Forming RAG database

## Dependencies installation

In [4]:
# @title Installation assuming Colab environment
!pip -q install -U pip
!pip -q install "lightrag-hku>=1.2.5" "raganything>=0.1.7" "docling>=2.8.0" "openai>=1.40.0" pillow tqdm python-dotenv
# Optional alternate parser for OCR fallback:
# !pip -q install "mineru>=0.1.7"


In [22]:
# @title Imports and path specification
from google.colab import userdata
import asyncio, os, shutil
from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc


API_KEY=userdata.get('api-key-MicRisk')
folder_path='/content/documents'


# ---- paths & models for the database storage ----
WORKING_DIR = "rag_storage"
OUTPUT_DIR  = "output"
TEXT_MODEL   = "gpt-4o"
VISION_MODEL = "gpt-4o"
EMBED_MODEL  = "text-embedding-3-large"


## Forming GraphDatabase using LightRAG

In [ ]:
def clear_caches():
    shutil.rmtree(WORKING_DIR, ignore_errors=True)
    shutil.rmtree(OUTPUT_DIR,  ignore_errors=True)
    os.makedirs(WORKING_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR,  exist_ok=True)

def make_llm_funcs(api_key: str, base_url: str | None):
    def llm_model_func(prompt, system_prompt=None, history_messages=None, **kwargs):
        return openai_complete_if_cache(
            TEXT_MODEL,
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages or [],
            api_key=api_key,
            base_url=base_url,
            **kwargs,
        )

    def vision_model_func(prompt, system_prompt=None, history_messages=None, image_data=None, messages=None, **kwargs):
        if messages:
            result = openai_complete_if_cache(
                VISION_MODEL, "", system_prompt=None, history_messages=[],
                messages=messages, api_key=api_key, base_url=base_url, **kwargs
            )
            return result if result is not None else ""  # Return empty string if result is None
        elif image_data:
            mm = []
            if system_prompt:
                mm.append({"role": "system", "content": system_prompt})
            mm.append({
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                ],
            })
            result = openai_complete_if_cache(
                VISION_MODEL, "", system_prompt=None, history_messages=[],
                messages=mm, api_key=api_key, base_url=base_url, **kwargs
            )
            return result if result is not None else "" # Return empty string if result is None
        else:
            return llm_model_func(prompt, system_prompt, history_messages, **kwargs)


    embedding_func = EmbeddingFunc(
        embedding_dim=3072,
        max_token_size=8192,
        func=lambda texts: openai_embed(
            texts, model=EMBED_MODEL, api_key=api_key, base_url=base_url
        ),
    )
    return llm_model_func, vision_model_func, embedding_func

async def run_docling(pdf_path: str):
    api_key  = API_KEY
    base_url = os.environ.get("OPENAI_BASE_URL")  # optional
    assert api_key, "Missing OPENAI_API_KEY."

    clear_caches()

    config = RAGAnythingConfig(
        working_dir=WORKING_DIR,
        parser="docling",          # robust figure+caption extraction
        parse_method="auto",       # use "ocr" for scanned PDFs
        enable_image_processing=True,
        enable_table_processing=True,
        enable_equation_processing=True,
    )

    llm_fn, vis_fn, emb_fn = make_llm_funcs(api_key, base_url)
    rag = RAGAnything(config=config, llm_model_func=llm_fn, vision_model_func=vis_fn, embedding_func=emb_fn)

    print(">> Parsing PDF with Docling ...")
    try:
        #await rag.process_document_complete(file_path=pdf_path, output_dir=OUTPUT_DIR, parse_method="auto")
        await rag.process_folder_complete(folder_path=pdf_path,output_dir=OUTPUT_DIR, file_extensions=[".pdf", ".docx", ".pptx"], recursive=True, max_workers=4,
                                           parse_method="auto")
    except asyncio.CancelledError:
        print(">> Document processing was cancelled.")
        return # Exit the function if cancelled


await asyncio.sleep(0)  # make sure event loop is ready in Colab
await run_docling(folder_path)

INFO: RAGAnything initialized with config:
INFO:   Working directory: rag_storage
INFO:   Parser: docling
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1


>> Parsing PDF with Docling ...


INFO: Parser 'docling' installation verified
INFO: Initializing LightRAG with parameters: {'working_dir': 'rag_storage'}
INFO: [_] Created new empty graph file: rag_storage/graph_chunk_entity_relation.graphml
INFO: [_] Process 1132 KV load full_docs with 0 records
INFO: [_] Process 1132 KV load text_chunks with 0 records
INFO: [_] Process 1132 KV load full_entities with 0 records
INFO: [_] Process 1132 KV load full_relations with 0 records
INFO: [_] Process 1132 KV load entity_chunks with 0 records
INFO: [_] Process 1132 KV load relation_chunks with 0 records
INFO: [_] Process 1132 KV load llm_response_cache with 0 records
INFO: [_] Process 1132 doc status load doc_status with 0 records
INFO: [_] Process 1132 KV load parse_cache with 0 records
INFO: Multimodal processors initialized with context support
INFO: Available processors: ['image', 'table', 'equation', 'generic']
INFO: Context configuration: ContextConfig(context_window=1, context_mode='page', max_context_tokens=2000, include_

## Quering the formed database

`make_llm_funcs` and `API_KEY` settings are the same as at the database creation step. The corresponding code is still included to permit the quering
the existing database: user does not need to create it every time and can
reuse the existing materials stored in **WORKING_DIR** folder)


In [ ]:
# @title Importing dependencies and specifying quering models
import asyncio
import os
from raganything import RAGAnything, RAGAnythingConfig
from lightrag import LightRAG
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc
from google.colab import userdata

import pandas as pd # to process the datasets
import random # to generate examples from the database during few-shot calls

# ---- paths & models ----
WORKING_DIR  = "rag_storage"      # your existing storage folder
TEXT_MODEL   = "gpt-5-nano"
VISION_MODEL = "gpt-5-nano"
EMBED_MODEL  = "text-embedding-3-large"

# Set your API key & base URL
API_KEY=userdata.get('api-key-MicRisk')
base_url = os.environ.get("OPENAI_BASE_URL")

In [27]:
# @title Loading an existing RAG instance and define quering LLM

def make_llm_funcs(api_key: str, base_url: str | None):
    def llm_model_func(prompt, system_prompt=None, history_messages=None, **kwargs):
        return openai_complete_if_cache(
            TEXT_MODEL,
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages or [],
            api_key=api_key,
            base_url=base_url,
            **kwargs,
        )

    def vision_model_func(prompt, system_prompt=None, history_messages=None, image_data=None, messages=None, **kwargs):
        if messages:
            result = openai_complete_if_cache(
                VISION_MODEL,
                "",
                system_prompt=None,
                history_messages=[],
                messages=messages,
                api_key=api_key,
                base_url=base_url,
                **kwargs
            )
            return result if result is not None else ""
        elif image_data:
            mm = []
            if system_prompt:
                mm.append({"role": "system", "content": system_prompt})
            mm.append({
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}}
                ],
            })
            result = openai_complete_if_cache(
                VISION_MODEL,
                "",
                system_prompt=None,
                history_messages=[],
                messages=mm,
                api_key=api_key,
                base_url=base_url,
                **kwargs
            )
            return result if result is not None else ""
        else:
            return llm_model_func(prompt, system_prompt, history_messages, **kwargs)

    embedding_func = EmbeddingFunc(
        embedding_dim=3072,
        max_token_size=8192,
        func=lambda texts: openai_embed(
            texts, model=EMBED_MODEL, api_key=api_key, base_url=base_url
        ),
    )
    return llm_model_func, vision_model_func, embedding_func

async def load_existing_rag(api_key: str, base_url: str | None):
    # Setup LLM / embeddings
    llm_fn, vis_fn, emb_fn = make_llm_funcs(api_key, base_url)

    # Create the underlying LightRAG instance pointing to the existing folder
    lightrag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_fn,
        embedding_func=emb_fn
    )

    # IMPORTANT: initialize storages (and pipeline status if needed) so that existing storage is loaded. :contentReference[oaicite:0]{index=0}
    await lightrag.initialize_storages()
    # Depending on version: you may need to call `await initialize_pipeline_status()` here.
    # from lightrag.kg.shared_storage import initialize_pipeline_status
    # await initialize_pipeline_status()

    # Now wrap it in RAGAnything with vision capability
    rag = RAGAnything(
        lightrag = lightrag,
        vision_model_func = vis_fn
    )

    return rag

# Optionally: helper to query in one function
async def ask_question(rag, question: str, mode: str = "hybrid"):
    return await rag.aquery(question, mode=mode)


## Single term test

In [35]:
# @title Testing the definition formation for a single term


# Load the RAG instance — use await since we’re in notebook
rag = await load_existing_rag(api_key, base_url)

term ='Bacteria'

question = f"""Based on the literature in your database you need to provide a concise definition for the term '{term}'.
    Only provide the definition of the term, do not provide any other information such as References.
    If your databasse does not contain information about the term : {term}, then return "No definition" only.

     """

ans = await ask_question(rag, question, mode="hybrid")


INFO: [_] Loaded graph from rag_storage/graph_chunk_entity_relation.graphml with 2166 nodes, 1376 edges
INFO: RAGAnything initialized with config:
INFO:   Working directory: ./rag_storage
INFO:   Parser: mineru
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1
INFO: Parser 'mineru' installation verified
INFO: Initializing parse cache for pre-provided LightRAG instance
INFO: Multimodal processors initialized with context support
INFO: Available processors: ['image', 'table', 'equation', 'generic']
INFO: Context configuration: ContextConfig(context_window=1, context_mode='page', max_context_tokens=2000, include_headers=True, include_captions=True, filter_content_types=['text'])
INFO: Executing VLM enhanced query: Based on the literature in your database you need to provide a concise definition for the term 'Bact...
INFO: Query nodes: Bacteria, concise definition, term Bacteria, database literature (top_k:40

## KIDA dataset test


In [36]:
# @title Testing the definition formation for groupped terms

# --- config ---
input_xlsx  = "/content/Terms_wiki_test_agent.xlsx"   #
output_xlsx = "/content/Terms_wiki_test_agent_with_definitions.xlsx"

api_key  = API_KEY   #
base_url = os.environ.get("OPENAI_BASE_URL")


async def fill_definitions_from_rag():
    # Load RAG instance
    rag = await load_existing_rag(api_key, base_url)

    # Load Excel
    df = pd.read_excel(input_xlsx)

    # Make sure the column exists
    if "Definition" not in df.columns:
        df["Definition"] = ""

    # Iterate over terms
    for idx, row in df.iterrows():
        term = str(row.get("Term", "")).strip()
        if not term:
            continue  # skip empty terms

        # Build question for this term

        question = f"""Based on the literature in your database you need to provide a concise definition for the term '{term}'.
    Only provide the definition of the term, do not provide any other information such as References.
    If your databasse does not contain information about the term : {term}, then return "No definition" only."""

        # Ask LightRAG
        ans = await ask_question(rag, question, mode="hybrid")

        # Extract the answer text depending on what ask_question returns
        if isinstance(ans, dict):
            # common pattern: {"answer": "...", "contexts": [...]}
            definition = ans.get("answer", "")
        else:
            # if it's just a plain string
            definition = str(ans)

        definition = definition.strip()

        # Only write non-empty answers
        if definition and "no definition" not in definition.lower():
            df.at[idx, "Definition"] = definition
            df.at[idx, "Definition_type"] = 'Light_RAG'

    # Save to new Excel
    df.to_excel(output_xlsx, index=False)
    print(f"Saved updated file to: {output_xlsx}")


# Run in notebook
await fill_definitions_from_rag()


INFO: [_] Loaded graph from rag_storage/graph_chunk_entity_relation.graphml with 2166 nodes, 1376 edges
INFO: RAGAnything initialized with config:
INFO:   Working directory: ./rag_storage
INFO:   Parser: mineru
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1
INFO: Parser 'mineru' installation verified
INFO: Initializing parse cache for pre-provided LightRAG instance
INFO: Multimodal processors initialized with context support
INFO: Available processors: ['image', 'table', 'equation', 'generic']
INFO: Context configuration: ContextConfig(context_window=1, context_mode='page', max_context_tokens=2000, include_headers=True, include_captions=True, filter_content_types=['text'])
INFO: Executing VLM enhanced query: Based on the literature in your database you need to provide a concise definition for the term 'Aeti...
INFO: Query nodes: Aetiologic agent, concise definition, definition constraint, references, l

Saved updated file to: /content/Terms_wiki_test_agent_with_definitions.xlsx


## Few-shot call (Dabase as a context)

In [39]:
# @title Testing the definition formation for groupped terms  with Few-shot call (Dabase as a context)

# Duplicates for Active/Inactive terms are removed
df = pd.read_excel("/content/Terms_wiki_test_agent_definitions_no_duplicates.xlsx")


# Filter rows where Definition_type == "Light_RAG"
light_rag_rows = df[df["Definition_type"] == "Light_RAG"]

# Safety check
if light_rag_rows.empty:
    raise ValueError("No rows found with Definition_type = 'Light_RAG'.")

# Randomly select one row
sample_df = light_rag_rows.sample(n=min(5, len(light_rag_rows)))

# Build multiline string
examples = []
for i, row in enumerate(sample_df.itertuples(), start=1):
    term = row.Term
    definition = row.Definition
    examples.append(f"{i}) Term: {term}\n   Definition: {definition}")

# Save into a single variable later used as examples for a few shot call
definition_examples = "\n\n".join(examples)

In [42]:
import os
import asyncio
import pandas as pd

# --- config ---
input_xlsx  = "/content/Terms_wiki_test_agent_definitions_no_duplicates.xlsx"
output_xlsx = "/content/Terms_wiki_test_agent_definitions_no_duplicates_general_defs.xlsx"

api_key  = API_KEY
base_url = os.environ.get("OPENAI_BASE_URL")


async def fill_definitions_from_rag():
    # Load RAG instance
    rag = await load_existing_rag(api_key, base_url)

    # Load Excel
    df = pd.read_excel(input_xlsx)

    # Ensure required columns exist
    if "Definition" not in df.columns:
        df["Definition"] = ""
    if "Definition_type" not in df.columns:
        df["Definition_type"] = ""

    # Iterate **only** over rows where Definition_type is empty / NaN
    rows_to_process = df[df["Definition_type"].isna() | (df["Definition_type"].astype(str).str.strip() == "")]

    for idx, row in rows_to_process.iterrows():
        term = str(row.get("Term", "")).strip()
        if not term:
            continue  # skip empty terms

        # Build question for this term
        question = f"""Using the literature in your database as a context provide a concise definition for the term '{term}'.
    Only provide the definition of the term, do not provide any other information such as References.
    Below are some terms with the examples {definition_examples}"""


        # Ask LightRAG
        ans = await ask_question(rag, question, mode="hybrid")

        # Extract answer
        if isinstance(ans, dict):
            definition = ans.get("answer", "")
        else:
            definition = str(ans)


            df.at[idx, "Definition"] = definition
            df.at[idx, "Definition_type"] = "Few-shot call"

    # Save updated file
    df.to_excel(output_xlsx, index=False)
    print(f"Saved updated file to: {output_xlsx}")


# Run in notebook
await fill_definitions_from_rag()


INFO: [_] Loaded graph from rag_storage/graph_chunk_entity_relation.graphml with 2166 nodes, 1376 edges
INFO: RAGAnything initialized with config:
INFO:   Working directory: ./rag_storage
INFO:   Parser: mineru
INFO:   Parse method: auto
INFO:   Multimodal processing - Image: True, Table: True, Equation: True
INFO:   Max concurrent files: 1
INFO: Parser 'mineru' installation verified
INFO: Initializing parse cache for pre-provided LightRAG instance
INFO: Multimodal processors initialized with context support
INFO: Available processors: ['image', 'table', 'equation', 'generic']
INFO: Context configuration: ContextConfig(context_window=1, context_mode='page', max_context_tokens=2000, include_headers=True, include_captions=True, filter_content_types=['text'])
INFO: Executing VLM enhanced query: Using the literature in your database as a context provide a concise definition for the term 'Bacter...
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 

Saved updated file to: /content/Terms_wiki_test_agent_definitions_no_duplicates_general_defs.xlsx
